# movie recommendation system using user based collaborative filtering

in this notebook i built a simple movie recommendation system using user-based collaborative filtering. I first prepare the data, build a utility matrix, calculate similarities between users and then predict movie ratings. Finally i evaluate the model using RMSE and generate predictions for the test set.

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

## loading the dataset

In [ ]:
ratings = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv')
print(f"Full dataset: {ratings.shape[0]} ratings, {ratings['userId'].nunique()} users, {ratings['movieId'].nunique()} movies")

In [ ]:
# Keep only the most active users and the most-rated movies, so the
# utility matrix stays small enough to build densely.
N_USERS = 3000
N_MOVIES = 3000

top_users = ratings['userId'].value_counts().nlargest(N_USERS).index
top_movies = ratings['movieId'].value_counts().nlargest(N_MOVIES).index

sub_ratings = ratings[
    ratings['userId'].isin(top_users) & ratings['movieId'].isin(top_movies)
].reset_index(drop=True)

print(f"Subset: {len(sub_ratings)} ratings, {sub_ratings['userId'].nunique()} users, {sub_ratings['movieId'].nunique()} movies")

## validation set

In [ ]:
val_df = sub_ratings.sample(frac=0.1, random_state=42)
train_df = sub_ratings.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train:      {len(train_df)} ratings")
print(f"Validation: {len(val_df)} ratings")

## building the utility matrix

In [ ]:
util_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
util_matrix.shape

## finding similar users using cosine similarity

In [ ]:
# normalize each row (user's ratings)
util_matrix_norm = util_matrix.apply(lambda x: (x - np.mean(x)) / (np.max(x) - np.min(x)), axis=1)

# fill NaNs with 0, transpose for easier indexing, and drop users with no ratings
util_matrix_norm.fillna(0, inplace=True)
util_matrix_norm = util_matrix_norm.T
util_matrix_norm = util_matrix_norm.loc[:, (util_matrix_norm != 0).any(axis=0)]

# sparse representation to save memory
util_matrix_sparse = sp.sparse.csr_matrix(util_matrix_norm.values)

In [ ]:
user_similarity = cosine_similarity(util_matrix_sparse.T)
user_sim_df = pd.DataFrame(
    user_similarity,
    index=util_matrix_norm.columns,
    columns=util_matrix_norm.columns,
)
user_sim_df.iloc[:5, :5]

## predicting the ratings

In [ ]:
K = 20  # number of similar users to use per prediction

neighbor_cache = {}
for user_id in user_sim_df.columns:
    sims = user_sim_df[user_id].drop(index=user_id)
    neighbor_cache[user_id] = sims.nlargest(K)

global_mean = train_df['rating'].mean()
movie_means = train_df.groupby('movieId')['rating'].mean()

ratings_by_movie = {}
for row in train_df.itertuples():
    ratings_by_movie.setdefault(row.movieId, {})[row.userId] = row.rating


def predict_rating(user_id, movie_id):
    if user_id not in neighbor_cache:
        return movie_means.get(movie_id, global_mean)

    neighbors = neighbor_cache[user_id]        
    movie_ratings = ratings_by_movie.get(movie_id, {}) 
    weighted_sum = 0.0
    weight_total = 0.0
    for neighbor_id, similarity in neighbors.items():
        if neighbor_id in movie_ratings:
            weighted_sum += similarity * movie_ratings[neighbor_id]
            weight_total += similarity

    if weight_total == 0:
        return movie_means.get(movie_id, global_mean)

    predicted = weighted_sum / weight_total
    return float(np.clip(predicted, 1.0, 5.0))


def predict_batch(df):
    return [predict_rating(u, m) for u, m in zip(df['userId'], df['movieId'])]

## evaluating the model using RMSE.

In [ ]:
val_df = val_df.copy()
val_df['pred_rating'] = predict_batch(val_df)

rmse = np.sqrt(np.mean((val_df['pred_rating'] - val_df['rating']) ** 2))
print(f"RMSE: {rmse:.4f}")

## making predictions for the test set

In [ ]:
test_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv')
print(f"Test: {len(test_df)} pairs")

# How much of the real test set is covered by our subsampled model
users_covered = test_df['userId'].isin(train_df['userId']).mean()
movies_covered = test_df['movieId'].isin(train_df['movieId']).mean()
print(f"Test users:  {users_covered:.1%}")
print(f"Test movies: {movies_covered:.1%}")

test_df = test_df.copy()
test_df['rating'] = predict_batch(test_df)

submission = pd.DataFrame({
    'Id': test_df['userId'].astype(str) + '_' + test_df['movieId'].astype(str),
    'rating': test_df['rating'].round(2),
})
submission.to_csv('submission.csv', index=False)

print(f"Wrote submission.csv with {len(submission)} rows.")
submission.head()

# test 

In [ ]:
example = val_df.iloc[0]
print(f"User {int(example['userId'])}, Movie {int(example['movieId'])}")
print(f"Actual rating:    {example['rating']}")
print(f"Predicted rating: {example['pred_rating']:.2f}")